In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
from xhistogram.xarray import histogram as xhist

In [ ]:
ds = xr.open_dataset("../data/SV_gridParticles.zarr")
ds

In [ ]:
lon_diff_zero = ds.lon.diff(dim="obs").isel(obs=0) == 0
lat_diff_zero = ds.lat.diff(dim="obs").isel(obs=0) == 0
stuck = (lon_diff_zero & lat_diff_zero).squeeze().compute()

In [ ]:
ds_clean = ds.isel(trajectory=~stuck.values)
ds_clean

In [ ]:
bin_width = 0.01

lon_min = float(ds_clean.lon.min())
lon_max = float(ds_clean.lon.max())
lat_min = float(ds_clean.lat.min())
lat_max = float(ds_clean.lat.max())

lon_bins = np.arange(lon_min, lon_max + bin_width, bin_width)
lat_bins = np.arange(lat_min, lat_max + bin_width, bin_width)

In [ ]:
obs_counts = xhist(
    ds_clean.lon, ds_clean.lat,
    bins=[lon_bins, lat_bins],
    dim=["trajectory", ],
    bin_dim_suffix=""
)

obs_counts

In [ ]:
obs_counts.to_netcdf("012_heatmaps.nc")

In [ ]:
obs_counts.isel(obs=0).plot(x="lon", y="lat")

In [ ]:
obs_counts.pad(obs=(0, 1), constant_values=0).coarsen(obs=10, boundary="trim").sum().sum(dim="obs").plot(x="lon", y="lat")

In [ ]:
obs_counts.pad(obs=(0, 1), constant_values=0).coarsen(obs=10, boundary="trim").sum().plot(col="obs", col_wrap=6, x="lon", y="lat")